In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Regex based filter on original Chembl datasets 
#Filter and categorise to EGFR vs ADME , further EGFR to On Target and Off Target, We need On Target EGFR Only
#Main Reference Standard is dataset Osimertinib, to filter and expand other ligands

Suggested Dataset Filtering Guide:
1. Manually go through and do EDA on original chembl ligand dataset raw 
2. Understand each unique values and what they mean (theory)
3. Adjust the Regex fitler to match different ligand datasets for different TKI inhibitor generation 
4. Manually validate after filtering, filter off uncertainties manually 

2025 November , Main Reference Standard is dataset Osimertinib, to filter and expand other ligands


In [ ]:
df_osi_chembl = pd.read_csv('/mnt/d/Publications/project_insilico/dataset/nov_2025/osimertinib/osimertinib_nov_2025_chembl.csv')
df_osi_chembl.shape

In [ ]:
import pandas as pd
import numpy as np
import re

#Filter Guide (Mannually screen though csv columns first for each ligand)
#Rationale:
# Different ligands have different cells lines tested and different mutations, need to carefully filter which to include/exclude
# Steps:
#1. Target Name and Assay Description are dominant columns for pattern matching
#2. Key facts: Cell Line, Mutation, On-target vs off-target, Cytotoxicity/ADME-toxicity markers
#3. Capture and categorize based on hierarchical filtering
#4. Finaly manually validate filtered results to match relationship acitvity 

# Load your DataFrame
df = df_osi_chembl  # Replace with your actual dataframe

# Clean ALL columns: strip whitespace, lowercase, replace empty strings with NaN
for col in df.columns:
    if df[col].dtype == 'object':  # Only process string columns
        df[col] = df[col].str.strip().str.lower().replace('', np.nan)

# Convert Standard Value to numeric
df['Standard Value'] = pd.to_numeric(df['Standard Value'], errors='coerce')

# Fill NaN with empty string for pattern matching
df['Target Name'] = df['Target Name'].fillna('')
df['Assay Description'] = df['Assay Description'].fillna('')

# ===== PART A: EGFR ON-TARGET PATTERNS =====
egfr_patterns = [
    r'epidermal.*growth.*factor.*receptor',
    r'egfr', r'erbb1', r'erbB1',
    r'apoptosis', r'proliferation', r'cytotoxicity', r'antiproliferative', r'inihibition',
    r'anticancer', r'anti-cancer', r'antitumor',
    r'anti.*tumor', r'anti.*tumour',
    r'double.*mutant', r'triple.*mutant',
    r'egfr.*mutant', r'egfr.*variant',
    r'\b(?:tyrosine\s+)?kinase\s+domain',
    r'\btkd\b',
    r'\bkinase\s+domain',
    r'\bcatalytic\s+domain',
    r'\batp.*bind',
    r'activation.*loop',
    r'c.?lobe',
    r'n.?lobe',
    r'phosphorylation',
    r'autophosphorylation'
]

egfr_mutations = [
    r'l858r', r't790m', r'exon.*19', r'exon.*21',
    r'\b19\b', r'\b21\b', r'\b20\b',
    r'\b858\b', r'\b790\b', r'\b797\b', r'\b719\b', r'\b768\b', r'\b861\b', r'\b718\b',
    r'g719[sxac]', r'l861q', r's768i', r'c797[sx]',
    r'del19', r'e746.a750', r'l747.e749', r'l747.p753',
    r'l747.s752', r'l747.t751'
]

egfr_cell_lines = [
    r'a.?431', r'hcc4006', r'u.?87', r'u87.?mg', 
    r'erbB1', r'erbb1', r'hcc', r'nci', r'pc', r'calu', r'baf3', r'ba/f3',
    r'\b431\b', r'\b1975\b', r'\b3255\b', r'\b1650\b', r'\b827\b', r'\b4006\b', 
    r'\b2279\b', r'\b9er\b', r'\b1299\b', r'\b292\b', r'\b460\b',
    r'bid007', r'kyse270', r'kyse450',
    r'nci.?h1666', r'nci.?h1975', r'hcc827', r'nci.?h3255',
    r'pc.?9', r'nci.?h1650', r'cl97', r'calu.?3',
    r'nci.?h820', r'hcc827gr5', r'pc9.?brc1', r'nci.?h2279',
    r'dfci.?127'
]

# ===== PART B: CYTOTOXICITY / ADME-TOX =====
cyto_cells = [
    r'hepatocyte', r'cho', r'hk.?2', r'l02', r'huvec', r'bj', r'ea\.hy\.?926',
    r'beas.?2b', r'hepg2', r'ht.?29', r'kg.?1', r'lymphoma',
    r'mda.?mb.?468', r'mda.?mb.?231', r'vero', r'vero.?c1008',
    r'liver', r'primary.*hepatocyte', r'hep.*rg', r'hepatocytes', r'histone.*deacetylase',
    r'guinea.*pig', r'canine', r'dog', r'monkey', r'cynomolgus',
    r'a549', r'mcf7', r'sk.?br.?3', r't47d', r'bt.?474', r'hct.?116', r'lovo',
    r'dld.?1', r'sw480', r'sw.?620', r'panc.?1', r'ovcar.?8', r'sk.?ov.?3', r'kb',
    r'a101d', r'a498', r'nb69', r'rmg.?i', r'l5178y', r'u2os', r'saos.?2', r'sjsa.?1'
]

cyto_markers = [
    r'adme', r'admet',
    r'herg', r'potassium.*channel', r'cardiac.*ion.*channel', r'ion.*channel',
    r'cytochrome.*p450', r'cyp', r'cyp1a2', r'cyp2c9', r'cyp2d6', r'cyp2c8', r'cl97', r'lsd1',
    r'cyp3a4', r'cyp2b6', r'cyp2c19', r'cyp3a5', r'drug.*metabolism', r'mgh',
    r'monoamine.*oxidase', r'mao', r'cyclooxygenase', r'cox.?1', r'cox.?2',
    r'no.*relevant.*target', r'non.?protein.*target', r'dipeptidyl.*peptidase',
    r'molecular.*identity.*unknown', r'unspecific', r'off.?target', r'histone',
    r'beta.?1.*adrenergic.*receptor', r'beta.?2.*adrenergic.*receptor', r'prostaglandin.*receptor',
    r'type.?1.*angiotensin.*ii.*receptor', r'fibroblast.*growth.*factor.*receptor', r'gamma.*aminobutyric.*acid.*receptor',
    r'adenosine.*a1.*receptor', r'adenosine.*a2a.*receptor', r'adenosine.*a3.*receptor',
    r'5.?ht', r'serotonin.*receptor', r'dopamine.*receptor', r'histamine.*receptor',
    r'opioid.*receptor', r'sodium.*channel', r'calcium.*channel', r'gaba.*receptor',
    r'glutamate.*receptor', r'nmda.*receptor', r'ampa.*receptor', r'adregenic.*receptor',
]

# ===== PART C: OFF-TARGET ACTIVITY =====
other_activity_targets = [
    r'pi3.?kinase', r'pi3k', r'pi3', r'akt', r'mtor', r'phosphoinositide.*3.?kinase',
    r'tyrosine.*protein.*kinase.*abl', r'bcr.?abl', r'bcr/abl',
    r'stem.*cell.*growth.*factor.*receptor', r'c.?kit', r'\bkit\b',
    r'fibroblast.*growth.*factor.*receptor', r'fgfr1', r'fgfr2', r'fgfr3', r'fgfr4',
    r'hepatocyte.*growth.*factor.*receptor', r'\bmet\b', r'c.?met',
    r'tyrosine.*protein.*kinase.*receptor.*ret', r'\bret\b',
    r'tyrosine.*protein.*kinase.*receptor.*flt3', r'flt3',
    r'tyrosine.*protein.*kinase.*jak', r'jak1', r'jak2', r'jak3',
    r'eml4.?alk', r'\balk\b', r'anaplastic.*lymphoma.*kinase',
    r'vascular', r'vegf', r'pdgf', r'von',
    r'serine/threonine.*protein.*kinase.*b.?raf', r'braf', r'b.?raf', r'v600e',
    r'gak', r'chk1', r'chk2', r'plk', r'polo.*like.*kinase', r'aurora.*kinase',
    r'insulin.*receptor', r'insulin.*like.*growth.*factor.*i.*receptor', r'igf.?1r',
    r'vegf.*receptor', r'vegfr', r'pdgfr', r'src', r'syk', r'btk', r'tek', r'tie2',
    r'ros1', r'trk', r'ntrk', r'dna.*pk', r'mapk', r'erk', r'mek', r'raf', r'ras',
    r'kras', r'kras.*g12c', r'kras.*g12d', r'kras.*g12v', r'hras', r'nras',
    r'c.?src', r'lck', r'fyn', r'yes', r'fgr', r'lyn', r'hck', r'blk', r'yrk',
    r'fak', r'paxillin', r'p38', r'jnk', r'sapk', r'ikk', r'tbk1', r'ikkε', 
    r'androgen.*receptor', r'estrogen.*receptor', r'progesterone.*receptor', 
    r'beta.*adrenergic.*receptor', r'dopamine.*receptor',
    r'glucocorticoid.*receptor', r'mineralocorticoid.*receptor',
    r'peroxisome.*proliferator.*activated.*receptor', r'ppar',
    r'quinolone.*resistance.*protein.*nora', r'venezuelan.*equine.*encephalitis.*virus'
    r'cdc42', r'bc', r'cgmp', r'hacat', r'aspc1',
]

# Function to check pattern matches
def pattern_match(text, patterns):
    for pattern in patterns:
        if re.search(pattern, text):
            return True
    return False

# Function to categorize with hierarchical filtering
def categorize_hierarchical(target_name, assay_desc):
    """
    Returns: (category, has_egfr, has_cyto, has_off_target)
    Category 1: EGFR only (no cyto, no off-target)
    Category 2: Cyto/ADME
    Category 3: Off-target
    Category 0: Unclassified
    """
    # Check matches for all categories
    has_egfr = (pattern_match(target_name, egfr_patterns) or 
                pattern_match(target_name, egfr_mutations) or
                pattern_match(target_name, egfr_cell_lines) or
                pattern_match(assay_desc, egfr_patterns) or 
                pattern_match(assay_desc, egfr_mutations) or
                pattern_match(assay_desc, egfr_cell_lines))
    
    has_cyto = (pattern_match(target_name, cyto_cells) or
                pattern_match(target_name, cyto_markers) or
                pattern_match(assay_desc, cyto_cells) or
                pattern_match(assay_desc, cyto_markers))
    
    has_off_target = (pattern_match(target_name, other_activity_targets) or
                      pattern_match(assay_desc, other_activity_targets))
    
    # Hierarchical filtering: EGFR must not have cyto or off-target
    if has_egfr and not has_cyto and not has_off_target:
        return 1  # Pure EGFR on-target
    elif has_cyto:
        return 2  # Cytotoxicity/ADME-Tox
    elif has_off_target:
        return 3  # Off-target
    else:
        return 0  # Unclassified

# Apply categorization
df['Category'] = df.apply(lambda row: categorize_hierarchical(
    row['Target Name'], 
    row['Assay Description']
), axis=1)

# Create filtered DataFrames
df_egfr_pure = df[df['Category'] == 1].copy()
df_cyto = df[df['Category'] == 2].copy()
df_off_target = df[df['Category'] == 3].copy()
df_unclassified = df[df['Category'] == 0].copy()

# ===== FILTER BY STANDARD TYPE AND UNITS =====
# Filter for valid standard types
valid_types = ['ic50', 'ec50', 'ac50', 'gi50', 'activity']
df_egfr_filtered = df_egfr_pure[df_egfr_pure['Standard Type'].isin(valid_types)].copy()

# Filter for nM units only
df_egfr_filtered = df_egfr_filtered[df_egfr_filtered['Standard Units'] == 'nm'].copy()

# Save filtered files
df_egfr_pure.to_csv('lazer_egfr_pure_all.csv', index=False)
df_egfr_filtered.to_csv('lazer_egfr_filtered_final.csv', index=False)
df_cyto.to_csv('lazer_cyto.csv', index=False)
df_off_target.to_csv('lazer_off_target.csv', index=False)
df_unclassified.to_csv('lazer_unclassified.csv', index=False)

# Print comprehensive results
print("="*60)
print("FILTERING RESULTS")
print("="*60)

print("\n1. CATEGORICAL FILTERING:")
print(f"   Pure EGFR on-target (no cyto/off-target): {len(df_egfr_pure)}")
print(f"   Cytotoxicity/ADME-Tox: {len(df_cyto)}")
print(f"   Off-target activity: {len(df_off_target)}")
print(f"   Unclassified: {len(df_unclassified)}")

print("\n2. STANDARD TYPE/UNITS FILTERING:")
print(f"   After filtering for valid types & nM units: {len(df_egfr_filtered)}")
print(f"   Valid standard types found: {df_egfr_filtered['Standard Type'].value_counts().to_dict()}")

print("\n3. STANDARD VALUE STATISTICS:")
if len(df_egfr_filtered) > 0:
    print(f"   Mean: {df_egfr_filtered['Standard Value'].mean():.2f} nM")
    print(f"   Median: {df_egfr_filtered['Standard Value'].median():.2f} nM")
    print(f"   Min: {df_egfr_filtered['Standard Value'].min():.2f} nM")
    print(f"   Max: {df_egfr_filtered['Standard Value'].max():.2f} nM")
    print(f"   Std Dev: {df_egfr_filtered['Standard Value'].std():.2f} nM")

print("\n4. FILES SAVED:")
print("   - lazer_egfr_pure_all.csv (all pure EGFR entries)")
print("   - lazer_egfr_filtered_final.csv (EGFR + type/units filtered)")
print("   - lazer_cyto.csv")
print("   - lazer_off_target.csv")
print("   - lazer_unclassified.csv")
print("="*60)

# Drop temporary columns from original dataframe
df.drop(['Category'], axis=1, inplace=True)

In [11]:
import pandas as pd
import numpy as np

# Load your DataFrame
df = pd.read_csv('/mnt/d/Publications/project_insilico/dataset/nov_2025/osimertinib/osi_egfr_filtered_final.csv')

# Clean column names: strip whitespace and lowercase
df.columns = df.columns.str.strip().str.lower()

# Clean ALL column values: strip whitespace, lowercase, replace empty strings with NaN
for col in df.columns:
    if df[col].dtype == 'object':  # Only process string columns
        df[col] = df[col].str.strip().str.lower().replace('', np.nan)

# Fill NaN values in assay description with empty string for filtering
df['assay description'] = df['assay description'].fillna('')

# Define EGFR mutations pattern
egfr_mutations = [
    r'l858r', r't790m', r'exon.*19', r'exon.*21', r'del', r'797', r'exon.*20', 
    r'wt', r'wild', r'ins', r'sub',
    r'\b858\b', r'\b790\b', r'\b797\b', r'\b719\b', r'\b768\b', r'\b861\b', r'\b718\b',
    r'double', r'triple'
]

# Create regex pattern to match any of the mutations
pattern = '|'.join(egfr_mutations)

# Filter rows where 'assay description' contains any of the EGFR mutations
df_filtered = df[df['assay description'].str.contains(pattern, case=False, na=False, regex=True)]

# Save filtered DataFrame to CSV
df_filtered.to_csv('df_final_muts_assays_description.csv', index=False)

print(f"Original rows: {len(df)}")
print(f"Filtered rows (with EGFR mutations): {len(df_filtered)}")
print(f"Rows removed: {len(df) - len(df_filtered)}")


Original rows: 597
Filtered rows (with EGFR mutations): 578
Rows removed: 19
